## 1 - import packages

In [1]:
import numpy as np
np.random.seed(42)
from scipy.interpolate import RBFInterpolator
from sklearn.metrics import mean_squared_error

from pygem import FFD
import meshio

from tqdm import trange
import time

import concurrent.futures
from numba import njit

## 2 - read **rod** and **shroud** data and transfer data to np.ndarray

In [4]:
start_time = time.time()

In [2]:
def parse_points_content(content):
    """
    parse points from content
    :param content: file contents
    :return: points coordinates as numpy.array
    """

    def num_points(content):
        for idx, line in enumerate(content):
            try:
                return int(line), idx + 2
            except ValueError:
                continue

    num_points, start_idx = num_points(content)
    print("num_points: ", num_points, ", start_idx: ", start_idx + 1)

    # Extract relevant lines containing coordinates
    string_coords = content[start_idx : start_idx + num_points]

    # Join all lines and replace unwanted characters once
    joined_coords = (
        b" ".join(string_coords)
        .replace(b")", b"")
        .replace(b"(", b"")
        .replace(b"\n", b"")
    )

    # Convert to a numpy array in one go
    data = np.fromstring(joined_coords, sep=" ", dtype=float).reshape(num_points, 3)

    return data

In [3]:
# Open points_ref file
with open("mesh_data/points_data/points_ref", "rb") as f:
    # with open("mesh_data/test_points", "rb") as f:
    points_ref = parse_points_content(f.readlines())

num_points:  5995038 , start_idx:  22


In [4]:
# read boundary points stl file
rod_points = meshio.read("mesh_data/input_data/rodWall.stl").points
shroud_points = meshio.read("mesh_data/input_data/channelWall.stl").points

## 3 - Using **PyGem** to deform the mesh. The write_points and also save to vtk file 
### 3.1 - create **FFD** object. Set **FFD** parameters (ctr, weights), as well as NUM_SAMPLES

In [15]:
# define a function to apply the FFD to a mesh,
# the variable of the function are: 1- ffd object , 2- displacement intensity,
#                                   3- weight matrix, 4- meshPoints

def apply_ffd(ffd, disp_intense, weight_matrix, meshPoints):

    ffd.array_mu_x = (
        disp_intense
        * np.random.uniform(-1, 1, size=ffd.array_mu_x.shape)
        * weight_matrix
    )
    ffd.array_mu_y = (
        disp_intense
        * np.random.uniform(-1, 1, size=ffd.array_mu_x.shape)
        * weight_matrix
    )

    new_p = ffd(meshPoints)

    # save the displacement of ffd control points, including array_mu_x, array_mu_y, array_mu_z.
    displacement_data = (
        np.array([ffd.array_mu_x, ffd.array_mu_y, ffd.array_mu_z]).reshape(3, -1).T
    )

    return new_p, displacement_data

## 4. Define data preprocess and RBF functions

In [7]:
def write_points_to_file(points, filename):
    header = r"""/*--------------------------------*- C++ -*----------------------------------*\
| =========                 |                                                 |
| \\      /  F ield         | OpenFOAM: The Open Source CFD Toolbox           |
|  \\    /   O peration     | Version:  v2106                                 |
|   \\  /    A nd           | Website:  www.openfoam.com                      |
|    \\/     M anipulation  |                                                 |
\*---------------------------------------------------------------------------*/
FoamFile
{
    version     2.0;
    format      ascii;
    arch        "LSB;label=32;scalar=64";
    class       vectorField;
    location    "constant/polyMesh";
    object      points;
}
// * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * //
"""

    ender = r"""
// ************************************************************************* //
    """

    with open(filename, "w") as f:
        output = []
        output.append(header + "\n\n")
        output.append(f"{points.shape[0]}\n")
        output.append("(\n")
        for point in points:
            output.append(f"({point[0]:.8e} {point[1]:.8e} {point[2]:.8e})\n")
        output.append(")\n")
        output.append(ender)
        f.write("".join(output))

In [9]:
def sort_data_add_index(data, index=False):
    # Add indices to the data
    if index:
        indices = np.arange(len(data)).reshape(-1, 1)
        data_with_indices = np.hstack((data, indices))
    else:
        data_with_indices = data

    # Sort data by z values (column index 2 is z)
    sorted_data_with_indices = data_with_indices[data_with_indices[:, 2].argsort()]

    return sorted_data_with_indices

In [8]:
def group_data_by_z(data, z_min, z_max, z_step, index=False):
    # Sort data by z values
    if index:
        sorted_data_with_indices = sort_data_add_index(data, index=True)
    else:
        sorted_data_with_indices = sort_data_add_index(data)

    z_value = np.arange(z_min, z_max + z_step, z_step)

    # List to store the grouped data
    groups = []

    # Iterate through the sorted data to group points based on z ranges
    current_group = []

    for z in z_value:
        current_group = sorted_data_with_indices[
            (z - 1.0e-6 < sorted_data_with_indices[:, 2])
            & (sorted_data_with_indices[:, 2] < z + 1.0e-6)
        ]

        if current_group.size > 0:
            groups.append(current_group)

    return groups

In [17]:
# RBF of each group of points
def rbf_group(boundary_group, points_group):
    # Create RBF interpolator
    rbf = RBFInterpolator(
        boundary_group[:, :3],
        boundary_group[:, 3:],
        # neighbors=100,
        kernel="linear",
        epsilon=1,
    )

    # Interpolate the z values
    new_points = rbf(points_group[:, :3])

    # Combine the x, y, and z values
    new_points = np.hstack((new_points, points_group[:, 3:]))

    return new_points

In [18]:
def parallel_rbf_group(original_boundary, deformed_boundary, points_ref):
    # Group points based on z values
    boundary_group = group_data_by_z(
        np.hstack((original_boundary, deformed_boundary)), 0, 0.328, 0.001
    )
    points_group = group_data_by_z(points_ref, 0, 0.328, 0.001, index=True)

    # Use concurrent.futures for parallel processing
    processed_groups = []
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [
            executor.submit(rbf_group, bg, pg)
            for bg, pg in zip(boundary_group, points_group)
        ]
        for future in concurrent.futures.as_completed(futures):
            processed_groups.append(future.result())

    # Reconstruct the original data order using the processed groups
    processed_data = np.zeros(points_ref.shape)
    for group in processed_groups:
        processed_data[group[:, 3].astype(int)] = group[:, :3]

    return processed_data

### 5. Define FFD parameters, carry out FFD and RBF interpolate the internal points

In [12]:
# set parameters for the FFD
box_size = np.array([0.0296, 0.0268, 0.328])

# create the FFD object, with control points 5x5x60
ffd = FFD([10, 10, 50])

ffd.box_origin = np.array([-box_size[0] / 2, -box_size[1] / 2, 0])
ffd.box_length = box_size

# define the weight matrix. The boundary control points are fixed
weights = np.zeros(ffd.array_mu_x.shape)
weights[1:-1, 1:-1, :] = 1

In [13]:
# define the number of mesh samples, and the displacement intensity for the control points
displacement_intensity = 0.15
NUM_SAMPLES = 1

In [19]:
all_displacement_data = None
all_boundary_points = None

original_points = np.concatenate((shroud_points, rod_points), axis=0)

for i in trange(NUM_SAMPLES):
    new_rod_points, displacement_data = apply_ffd(
        ffd, displacement_intensity, weights, rod_points
    )
    boundary_points = np.concatenate((shroud_points, new_rod_points), axis=0)
    # print(f"\t Time apply_ffd: {(time.time() - start_time)} seconds")

    # parallel_rbf_group to displace the points
    # points_ref_displaced = parallel_rbf_group(original_points, boundary_points, points_ref)
    points_ref_displaced = parallel_rbf_group(
        original_points, boundary_points, points_ref
    )
    # print(f"\t Time parallel_rbf_group: {(time.time() - start_time)} seconds")

    # write the points to file
    write_points_to_file(points_ref_displaced, f"mesh_data/points_data/points_{i}")

    if i == 0:
        all_displacement_data = displacement_data
        all_boundary_points = boundary_points
    else:
        all_displacement_data = np.concatenate((all_displacement_data, displacement_data))
        all_boundary_points = np.concatenate((all_boundary_points, boundary_points))

# write the data to npy file
np.save("mesh_data/parameter_data/displacement_data.npy", all_displacement_data)
np.save("mesh_data/parameter_data/boundary_points.npy", all_boundary_points)

  0%|          | 0/1 [00:08<?, ?it/s]


TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Cannot infer the type of variable 'current_group', have imprecise type: list(undefined)<iv=None>. 

For Numba to be able to compile a list, the list must have a known and
precise type that can be inferred from the other variables. Whilst sometimes
the type of empty lists can be inferred, this is not always the case, see this
documentation for help:

https://numba.readthedocs.io/en/stable/user/troubleshoot.html#my-code-has-an-untyped-list-problem


File "../../../../../tmp/ipykernel_12620/771353844.py", line 15:
<source missing, REPL/exec in use?>
